# UACB-604 · Lecture 1.4 — Genetic maps by simulation

> **Colab users:** this notebook uses **R**. Choose **Runtime → Change runtime type → R** before running.

In the lecture we derived **Haldane's map function** from one assumption: crossovers fall along a chromosome like a Poisson process.
Here we *simulate* meioses, measure recombination fractions the way a geneticist would, and check the theory.

**You will:**
1. Simulate gametes from a heterozygous parent
2. Estimate the recombination fraction $r$ between markers
3. Compare $r$ against true distance with the Haldane and Kosambi map functions
4. Recover the **order** of shuffled markers (a small optimisation problem)
5. Add crossover **interference** and see which map function fits better

Only base R is used.

In [ ]:
set.seed(604)

# One meiosis: crossover positions (cM) on a chromosome of length L, Poisson with rate 1 per 100 cM.
crossovers_poisson <- function(L) sort(runif(rpois(1, L / 100), 0, L))

# A gamete: which parental haplotype (0 or 1) each marker comes from.
# Start on a random haplotype, switch at every crossover.
gamete <- function(markers, xo) {
  start <- rbinom(1, 1, 0.5)
  switches <- findInterval(markers, xo)       # number of crossovers to the left of each marker
  (start + switches) %% 2
}

simulate <- function(n, markers, L, xo_fun = crossovers_poisson) {
  t(replicate(n, gamete(markers, xo_fun(L))))
}

markers <- c(0, 5, 12, 20, 35, 50, 70, 95, 130)     # true positions in cM
names(markers) <- paste0("M", seq_along(markers))
G <- simulate(2000, markers, L = 150)
dim(G); head(G)

## 1. Recombination fraction
A gamete is **recombinant** for markers $i$ and $j$ if they come from different parental haplotypes. So
$\hat r_{ij}$ = fraction of gametes where column $i$ ≠ column $j$.

In [ ]:
r_hat <- function(G) {
  m <- ncol(G); R <- matrix(0, m, m, dimnames = list(colnames(G), colnames(G)))
  for (i in 1:m) for (j in 1:m) R[i, j] <- mean(G[, i] != G[, j])
  R
}
colnames(G) <- names(markers)
R <- r_hat(G)
round(R, 3)

## 2. Map functions
- **Haldane** (no interference): $r = \tfrac12\left(1 - e^{-2d}\right)$, so $d = -\tfrac12\ln(1-2r)$
- **Kosambi** (some interference): $r = \tfrac12\tanh(2d)$, so $d = \tfrac14\ln\dfrac{1+2r}{1-2r}$

($d$ in Morgans; 1 M = 100 cM.)

In [ ]:
haldane_r  <- function(d) 0.5 * (1 - exp(-2 * d))
kosambi_r  <- function(d) 0.5 * tanh(2 * d)
haldane_d  <- function(r) -0.5 * log(1 - 2 * r)

D  <- abs(outer(markers, markers, "-")) / 100        # true distances in Morgans
ut <- upper.tri(D)
plot(D[ut] * 100, R[ut], pch = 19, col = "slateblue", xlab = "true distance (cM)", ylab = "estimated r",
     main = "Recombination fraction vs distance (Poisson crossovers)", ylim = c(0, 0.55))
dd <- seq(0, 1.4, length.out = 200)
lines(dd * 100, haldane_r(dd), col = "red", lwd = 2)
lines(dd * 100, kosambi_r(dd), col = "darkgreen", lwd = 2, lty = 2)
lines(dd * 100, dd, col = "grey50", lty = 3)
abline(h = 0.5, col = "grey70")
legend("bottomright", c("Haldane", "Kosambi", "r = d (only near 0)"), col = c("red", "darkgreen", "grey50"),
       lty = c(1, 2, 3), lwd = 2, bty = "n")

✏️ **Exercise 1.** Why does $r$ never exceed 0.5, however far apart two markers are? Explain in one sentence using "odd and even numbers of crossovers".

## 3. Distances add; recombination fractions don't
Take three neighbouring markers A–B–C.

In [ ]:
trip <- c("M3", "M4", "M5")
r_ab <- R[trip[1], trip[2]]; r_bc <- R[trip[2], trip[3]]; r_ac <- R[trip[1], trip[3]]
cat(sprintf("r_AB + r_BC = %.3f   but observed r_AC = %.3f\n", r_ab + r_bc, r_ac))
cat(sprintf("Haldane: d_AB + d_BC = %.3f M,  d_AC = %.3f M\n",
            haldane_d(r_ab) + haldane_d(r_bc), haldane_d(r_ac)))
cat(sprintf("no-interference prediction r_AB + r_BC - 2 r_AB r_BC = %.3f\n", r_ab + r_bc - 2 * r_ab * r_bc))

## 4. Put shuffled markers back in order
In a real experiment you don't know the order of markers along the chromosome. One approach: choose the order that makes the **total map length** (sum of Haldane distances between neighbours) as small as possible.
With 6 markers there are 6!/2 = 360 distinct orders, so we can simply try them all.

In [ ]:
perms <- function(v) {                     # all permutations of a vector (base R, recursive)
  if (length(v) <= 1) return(list(v))
  out <- list()
  for (i in seq_along(v)) for (p in perms(v[-i])) out[[length(out) + 1]] <- c(v[i], p)
  out
}

six <- sample(names(markers)[1:6])          # shuffled labels: pretend we don't know the order
cat("shuffled:", six, "\n")
Dh <- haldane_d(pmin(R, 0.499))
len <- function(o) sum(Dh[cbind(o[-length(o)], o[-1])])
all_orders <- perms(six)
best <- all_orders[[which.min(sapply(all_orders, len))]]
cat("best order:", best, sprintf("(total %.1f cM)\n", 100 * len(best)))

✏️ **Exercise 2.** How many orders would you need to check for 20 markers? (Compute $20!/2$.) Is brute force still possible? This is a version of the **travelling salesman problem**, a famous hard problem. Suggest a greedy shortcut.

In [ ]:
# your code here

## 5. Interference
Real crossovers avoid each other: a crossover makes another one nearby less likely. A crude model: drop any crossover within 25 cM of the previous one.
Which map function fits better now?

In [ ]:
crossovers_interfere <- function(L, gap = 25) {
  xo <- sort(runif(rpois(1, 1.4 * L / 100), 0, L))   # raise the rate a bit, then thin
  keep <- numeric(0)
  for (x in xo) if (length(keep) == 0 || x - tail(keep, 1) >= gap) keep <- c(keep, x)
  keep
}
G2 <- simulate(2000, markers, L = 150, xo_fun = crossovers_interfere); colnames(G2) <- names(markers)
R2 <- r_hat(G2)

# Each map function predicts r from the *map* distance. Estimate map distance from tightly linked pairs (r ≈ d),
# then compare predictions for all pairs.
scale <- sum(R2[ut][D[ut] <= 0.08]) / sum(D[ut][D[ut] <= 0.08])   # crossover rate per Morgan of physical spacing
cat(sprintf("rate scale = %.2f\n", scale))
err <- function(f) sqrt(mean((R2[ut] - f(scale * D[ut]))^2))
cat(sprintf("RMS error  Haldane: %.3f   Kosambi: %.3f\n", err(haldane_r), err(kosambi_r)))

plot(scale * D[ut] * 100, R2[ut], pch = 19, col = "slateblue", xlab = "map distance (cM)", ylab = "estimated r",
     main = "With interference", ylim = c(0, 0.55))
lines(dd * 100, haldane_r(dd), col = "red", lwd = 2)
lines(dd * 100, kosambi_r(dd), col = "darkgreen", lwd = 2, lty = 2)
legend("bottomright", c("Haldane", "Kosambi"), col = c("red", "darkgreen"), lty = 1:2, lwd = 2, bty = "n")

✏️ **Exercise 3.** Change `gap` to 0 (no interference) and to 40. How do the two errors change? Write two sentences on what a map function *assumes*.

---
**Further reading:** Brown, *Genomes 5*, ch. 3 (mapping genomes: genetic and physical techniques).